# Auslan → English: Arm B/C training on Colab

This notebook is the current main experiment after the official Stage3 Arm A seed-0 reference. It trains Arm B in the fixed two-stage order and Arm C as three independent auxiliary-loss runs. Official Stage3 seed 1 and the custom 1e-4 Arm A ablation are deferred.

Run cells 1–8 in every new Colab session. Run the B1, B2, and C cells separately when GPU time is limited; each training cell uses resume and writes checkpoints to Drive. After a disconnect, rerun cells 1–8 and then rerun the interrupted training cell.

The joint manifest is required here: it contains Auslan-Daily and MM-WLAuslan. The notebook stops before training if the pose count is incomplete or if excluded.txt is not the 131-entry Auslan-Daily gate list.

## 1. Runtime check

The real Uni-Sign model needs a CUDA GPU. This notebook is intended for the A100 Colab runtime.

In [2]:
import copy
import glob
import hashlib
import json
import os
import shutil
import signal
import subprocess
import sys
import tarfile
import time

gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True).stdout.strip()
print(gpu or 'NO GPU')
if not gpu:
    raise RuntimeError('CUDA GPU is required; do not start a CPU run.')
DEVICE = 'cuda'

NVIDIA A100-SXM4-40GB, 40960 MiB


## 2. Dependencies

In [3]:
!pip -q install einops sacrebleu pyyaml
import torch
import transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA is not available.'

torch 2.11.0+cu128 | transformers 5.16.1 | cuda True


## 3. Mount Drive and locate the joint experiment data

In [4]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
WORK = f'{DRIVE}/auslan_work'
MANIFEST = f'{WORK}/manifest.jsonl'
EXCLUDE = f'{WORK}/excluded.txt'
for path in (MANIFEST, EXCLUDE):
    if not os.path.exists(path):
        raise SystemExit(f'{path} is missing. Run the setup notebook first.')
print('work dir:', WORK)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
work dir: /content/drive/MyDrive/auslan_work


## 4. Load the verified Uni-Sign code

In [5]:
CODE = '/content/unisign'

def locate(parts):
    for root in (DRIVE, '/content'):
        for prefix in ('', '*/', '*/*/'):
            hits = glob.glob(os.path.join(root, prefix, *parts))
            if hits:
                return hits[0]
    return None

src = locate(['unisign', 'spec.py'])
if src:
    src = os.path.dirname(src)
else:
    tarball = locate(['unisign_code.tar.gz'])
    if tarball is None:
        raise SystemExit('Upload the current unisign/ folder or unisign_code.tar.gz to Drive.')
    with tarfile.open(tarball) as tf:
        tf.extractall('/content/_bc_code')
    src = '/content/_bc_code/unisign'

if os.path.abspath(src) != CODE:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(src, CODE)
sys.path.insert(0, CODE)
import spec
print('code from', src)
print('spec fingerprint:', spec.SCHEMA_FINGERPRINT)
assert spec.SCHEMA_FINGERPRINT == 'bc3bb2df0f22948d', 'spec.py does not match the extracted poses.'

code from /content/drive/MyDrive/unisign
spec fingerprint: bc3bb2df0f22948d


## 5. Restore all poses and run the B/C data gate

B/C uses the joint manifest and therefore needs both Auslan-Daily and MM-WLAuslan poses. Run the two code cells below in order. The first restores poses and rebuilds a protected Auslan-Daily gate list; the second verifies the active list before training.

In [6]:
from collections import Counter
from pathlib import Path
from manifest import read_manifest

# Load the joint manifest before using rows/POSE_LOCAL. This makes this cell
# safe to run by itself after the code-loading cell.
rows = read_manifest(Path(MANIFEST))
counts = Counter((r['dataset'], r['split'], r.get('subset')) for r in rows)
for key, value in sorted(counts.items()):
    print(key, value)
n_ad = sum(r['dataset'] == 'auslandaily' for r in rows)
n_mmwl = sum(r['dataset'] == 'mmwlauslan' for r in rows)
assert n_ad == 25109, f'Expected 25109 Auslan-Daily rows, got {n_ad}'
assert n_mmwl == 51440, f'Expected 51440 MM-WLAuslan rows, got {n_mmwl}'

# Restore all joint poses to local Colab storage.
POSE_LOCAL = '/content/pose'
os.makedirs(POSE_LOCAL, exist_ok=True)
present = {os.path.basename(p)[:-4] for p in glob.glob(f'{POSE_LOCAL}/*.npz')}
if len(present) < len(rows):
    chunks = sorted(glob.glob(f'{WORK}/pose/chunk_*.tar'))
    if not chunks:
        raise SystemExit('No pose tar chunks found in Drive.')
    print(f'unpacking {len(chunks)} pose chunk(s) ...')
    for chunk in chunks:
        with tarfile.open(chunk) as tf:
            try:
                tf.extractall(POSE_LOCAL, filter='data')
            except TypeError:
                tf.extractall(POSE_LOCAL)
present = {os.path.basename(p)[:-4] for p in glob.glob(f'{POSE_LOCAL}/*.npz')}
missing = [r['uid'] for r in rows if r['uid'] not in present]
print(f'{len(rows)} manifest rows | {len(present)} poses | {len(missing)} missing')
if missing:
    raise SystemExit(f'{len(missing)} poses are missing, e.g. {missing[:3]}')

# Rebuild the AD gate from AD poses only, then keep a protected copy.
# The separate MM-WLAuslan checker may otherwise overwrite excluded.txt.
ad_rows = [r for r in rows if r['dataset'] == 'auslandaily']
AD_POSE_LOCAL = '/content/pose_auslan_daily'
AD_EXCLUDE = f'{WORK}/excluded_auslan_daily.txt'
os.makedirs(AD_POSE_LOCAL, exist_ok=True)
for r in ad_rows:
    src = f"{POSE_LOCAL}/{r['uid']}.npz"
    dst = f"{AD_POSE_LOCAL}/{r['uid']}.npz"
    if not os.path.exists(src):
        raise FileNotFoundError(src)
    if os.path.islink(dst) and not os.path.exists(dst):
        os.unlink(dst)
    if not os.path.lexists(dst):
        os.symlink(src, dst)

cmd = [
    sys.executable, 'verify_pose.py',
    '--npz-dir', AD_POSE_LOCAL,
    '--csv', f'{WORK}/quality_auslan_daily.csv',
    '--exclude-out', AD_EXCLUDE,
]
res = subprocess.run(cmd, cwd=CODE, capture_output=True, text=True)
print(res.stdout)
print(res.stderr)
if res.returncode != 0:
    raise RuntimeError(f'Auslan-Daily gate failed with code {res.returncode}')

all_uids = {r['uid'] for r in rows}
ad_excluded = {line.strip() for line in open(AD_EXCLUDE)
               if line.strip() and not line.startswith('#')}
if (
        len(ad_excluded) != 131
        or not ad_excluded <= all_uids
        or any(not uid.startswith('ad-') for uid in ad_excluded)
    ):
    raise RuntimeError(
        f'Auslan-Daily gate produced {len(ad_excluded)} UIDs; '
        'expected exactly 131 ad-* UIDs.')
shutil.copyfile(AD_EXCLUDE, EXCLUDE)
print(f'Restored protected AD gate: {AD_EXCLUDE}')
print(f'Active training exclude: {EXCLUDE}')

('auslandaily', 'test', 'communication') 800
('auslandaily', 'test', 'news') 700
('auslandaily', 'train', 'communication') 12440
('auslandaily', 'train', 'news') 9669
('auslandaily', 'val', 'communication') 800
('auslandaily', 'val', 'news') 700
('mmwlauslan', 'test', 'studio') 6430
('mmwlauslan', 'train', 'studio') 38580
('mmwlauslan', 'val', 'studio') 6430
76549 manifest rows | 76549 poses | 0 missing

=== layer 1: consistency ===
  OK  spec=unisign-pose-v2-verified fp=bc3bb2df0f22948d
      model=rtmlib.Wholebody/lightweight backend=onnxruntime rtmlib=0.0.16 person=largest

=== layer 2: numbers ===
  clips=25109
  auslandaily/communication  n= 14040  dominant-hand present median=1.000
  auslandaily/news           n= 11069  dominant-hand present median=1.000
  low_body                      172  (0.7%)
  MIRRORED_OR_BACK_VIEW         131  (0.5%)  <-- investigate
  low_face                       61  (0.2%)
  out_of_bounds                  49  (0.2%)
  low_dominant_hand              40 

In [7]:
from pathlib import Path
from manifest import read_manifest

# Final, non-destructive gate. If another notebook has overwritten the
# active file, restore it from the protected AD-only copy made above.
rows = read_manifest(Path(MANIFEST))
all_uids = {r['uid'] for r in rows}
AD_EXCLUDE = f'{WORK}/excluded_auslan_daily.txt'

def read_ids(path):
    return {line.strip() for line in open(path)
            if line.strip() and not line.startswith('#')}

excluded = read_ids(EXCLUDE)
protected = read_ids(AD_EXCLUDE) if os.path.exists(AD_EXCLUDE) else set()
if (len(excluded) != 131 or any(not uid.startswith('ad-') for uid in excluded)
        or not excluded <= all_uids):
    if len(protected) != 131 or any(not uid.startswith('ad-') for uid in protected):
        raise RuntimeError(
            f'No valid AD gate list: active={len(excluded)}, '
            f'protected={len(protected)}. Run the first cell in section 5.')
    shutil.copyfile(AD_EXCLUDE, EXCLUDE)
    excluded = set(protected)
    print('Active excluded.txt was repaired from the protected AD gate.')

ad_excluded = {uid for uid in excluded if uid.startswith('ad-')}
non_ad_excluded = excluded - ad_excluded
if len(excluded) != 131 or len(ad_excluded) != 131 or non_ad_excluded or not excluded <= all_uids:
    raise RuntimeError(
        f'excluded.txt is not the Auslan-Daily 131-entry gate list: '
        f'total={len(excluded)}, ad={len(ad_excluded)}, non_ad={len(non_ad_excluded)}.')
ad_train_val = {r['uid'] for r in rows if r['dataset'] == 'auslandaily' and r['split'] in ('train', 'val')}
ad_test = {r['uid'] for r in rows if r['dataset'] == 'auslandaily' and r['split'] == 'test'}
print(f'{len(rows)} manifest rows | {len(all_uids)} manifest UIDs')
print(f'excluded: {len(excluded)} total | {len(excluded & ad_train_val)} train/val | {len(excluded & ad_test)} test')
print('B/C data gate passed.')

76549 manifest rows | 76549 manifest UIDs
excluded: 131 total | 119 train/val | 12 test
B/C data gate passed.


## 6. Download and verify the real model

In [8]:
from huggingface_hub import hf_hub_download, snapshot_download

INIT_CKPT = 'csl_stage1_weight.pth'
REPO_DIR = '/content/Uni-Sign'
REPO_COMMIT = 'eed438bcb49e30405cd6ccdfcccca330c134e830'
MT5_DIR = f'{REPO_DIR}/pretrained_weight/mt5-base'
MT5_REVISION = '2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f'
UNISIGN_REVISION = 'eab251b7fe7e8521afc0e67be98add670ea40a0d'
SHA256 = {
    'csl_stage1_weight.pth': '3c81cf4a087e9e81581e57a2f33f8f0acf87b518a1ada540a660a76cdb144ced',
    'mt5-base/pytorch_model.bin': '180573b534144580f04af026da62bf71bc976ee1b7eb311b8945e2fefde8d614',
}

if not os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ZechengLi19/Uni-Sign.git', REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', REPO_COMMIT], check=True)
snapshot_download('google/mt5-base', revision=MT5_REVISION, local_dir=MT5_DIR, allow_patterns=['*.json', '*.model', 'pytorch_model.bin'])
CKPT = hf_hub_download('ZechengLi19/Uni-Sign', INIT_CKPT, revision=UNISIGN_REVISION, local_dir='/content/checkpoints')

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as fh:
        for block in iter(lambda: fh.read(1 << 24), b''):
            digest.update(block)
    return digest.hexdigest()

for name, path in ((INIT_CKPT, CKPT), ('mt5-base/pytorch_model.bin', f'{MT5_DIR}/pytorch_model.bin')):
    if sha256(path) != SHA256[name]:
        raise SystemExit(f'{name}: sha256 mismatch')
    print('OK', name)
print('Uni-Sign code at', REPO_COMMIT[:7])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


OK csl_stage1_weight.pth
OK mt5-base/pytorch_model.bin
Uni-Sign code at eed438b


## 7. Generate separate configs and run directories

B1, B2, and each C weight get a distinct output directory. Run the first code cell in this section; it accepts config names with or without the .yaml suffix. Do not reuse the old Arm A folders. B2 receives B1 only through its init-from checkpoint; C always starts from the CSL pose-only checkpoint.

In [9]:
import yaml

BASE = 'arm_bc__csl_stage1_weight'
TRAIN_PRECISION = 'bf16'
RUN_B1 = f'{BASE}__b1_pose'
RUN_B2 = f'{BASE}__b2_daily'
C_WEIGHTS = [0.1, 0.2, 0.3]
C_WEIGHTS_TO_RUN = C_WEIGHTS

def config_template(config_name):
    # Accept either 'arm_b_stage1' or 'arm_b_stage1.yaml'.
    stem = str(config_name).strip()
    if stem.lower().endswith('.yaml'):
        stem = stem[:-5]
    path = f'{CODE}/configs/{stem}.yaml'
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Config template not found: {path}')
    return path

def make_config(config_name, run_name):
    template = config_template(config_name)
    with open(template) as fh:
        cfg = yaml.safe_load(fh)
    cfg['num_workers'] = 8
    cfg['precision'] = TRAIN_PRECISION
    cfg['optim']['warmup_frac'] = 0.0
    cfg['data'].update(manifest=MANIFEST, npz_dir=POSE_LOCAL, exclude=EXCLUDE)
    cfg['backend'] = {
        'name': 'unisign',
        'checkpoint': CKPT,
        'repo': REPO_DIR,
        'mt5_path': MT5_DIR,
        'num_beams': 5,
        'max_new_tokens': 100,
        'label_smoothing': 0.2,
    }
    cfg['output_dir'] = f'{WORK}/runs/{run_name}'
    path = f'{WORK}/train_configs/{run_name}.yaml'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as fh:
        yaml.safe_dump(cfg, fh, sort_keys=False)
    return path, cfg

CFG_B1, CONFIG_B1 = make_config('arm_b_stage1.yaml', RUN_B1)
CFG_B2, CONFIG_B2 = make_config('arm_b_stage2.yaml', RUN_B2)

CFG_C = {}
CONFIG_C = {}
for weight in C_WEIGHTS:
    run_name = f'{BASE}__c__aux{weight:.1f}'
    path, cfg = make_config('arm_c.yaml', run_name)
    cfg['optim']['aux_weight'] = weight
    with open(path, 'w') as fh:
        yaml.safe_dump(cfg, fh, sort_keys=False)
    CFG_C[weight] = path
    CONFIG_C[weight] = cfg

OUT_B1 = CONFIG_B1['output_dir']
OUT_B2 = CONFIG_B2['output_dir']
OUT_C = {w: CONFIG_C[w]['output_dir'] for w in C_WEIGHTS}
print('B1:', CFG_B1, OUT_B1)
print('B2:', CFG_B2, OUT_B2)
for weight in C_WEIGHTS:
    print(f'C aux_weight={weight}:', CFG_C[weight], OUT_C[weight])
print('C weights selected for this session:', C_WEIGHTS_TO_RUN)
print('Training precision:', TRAIN_PRECISION)
print('Config templates generated without duplicate .yaml suffixes.')

B1: /content/drive/MyDrive/auslan_work/train_configs/arm_bc__csl_stage1_weight__b1_pose.yaml /content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__b1_pose
B2: /content/drive/MyDrive/auslan_work/train_configs/arm_bc__csl_stage1_weight__b2_daily.yaml /content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__b2_daily
C aux_weight=0.1: /content/drive/MyDrive/auslan_work/train_configs/arm_bc__csl_stage1_weight__c__aux0.1.yaml /content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__c__aux0.1
C aux_weight=0.2: /content/drive/MyDrive/auslan_work/train_configs/arm_bc__csl_stage1_weight__c__aux0.2.yaml /content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__c__aux0.2
C aux_weight=0.3: /content/drive/MyDrive/auslan_work/train_configs/arm_bc__csl_stage1_weight__c__aux0.3.yaml /content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__c__aux0.3
C weights selected for this session: [0.1, 0.2, 0.3]
Config templates generated without duplicate .yaml 

In [10]:
# Compatibility placeholder. Cell 7 now normalizes .yaml suffixes itself;
# do not regenerate the configs a second time.
print('Section 7 already generated all configs; no compatibility rerun needed.')

Section 7 already generated all configs; no compatibility rerun needed.


## 8. Training helpers

The helper streams logs and sends Ctrl-C to train.py so a manual Stop saves a resumable checkpoint. A completed run is skipped. Never remove a checkpoint to make a run restart; use a new run directory for a new experiment.

In [11]:
def launch(cfg_path, extra, log_path):
    cmd = [sys.executable, '-u', 'train.py', '--config', cfg_path, '--device', DEVICE] + list(extra)
    log = open(log_path, 'a')
    proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
    except KeyboardInterrupt:
        print('\nStop pressed: train.py will save a checkpoint before exiting ...', flush=True)
        proc.send_signal(signal.SIGINT)
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
    proc.wait()
    log.close()
    return proc.returncode

def train_one(label, cfg_path, out_dir, init_from=None, needs_metrics=False):
    complete = os.path.exists(f'{out_dir}/checkpoint.pt') and (not needs_metrics or os.path.exists(f'{out_dir}/metrics.json'))
    if complete:
        print(f'{label}: already finished; skipping {out_dir}')
        return True
    os.makedirs(out_dir, exist_ok=True)
    extra = ['--resume']
    if init_from:
        extra += ['--init-from', init_from]
    rc = launch(cfg_path, extra, f'{out_dir}/colab_train.log')
    if rc == 130:
        print(f'{label}: stopped safely; rerun this cell with --resume to continue.')
        return False
    elif rc != 0:
        raise RuntimeError(f'{label} exited with code {rc}')
    else:
        print(f'{label}: process finished.')
        return True

def eval_one(label, cfg_path, out_dir, tag, settings):
    dest = f'{out_dir}/eval_{tag}'
    if os.path.exists(f'{dest}/metrics.json'):
        print(f'{label} {tag}: already evaluated; skipping')
        return
    extra = ['--eval-only', '--eval-tag', tag]
    if settings:
        extra += ['--set'] + [f'{key}={value}' for key, value in settings.items()]
    rc = launch(cfg_path, extra, f'{out_dir}/colab_eval_{tag}.log')
    if rc != 0:
        raise RuntimeError(f'{label} {tag} exited with code {rc}')

## 8.1 B2 hyperparameter search grid

This cell only creates separate YAML files and prints the search plan; it does not start training. All new search and training configs use precision=bf16 and warmup_frac=0. First compare learning rates with effective batch size 8, then compare effective batch sizes using the selected learning rate. The completed B2 baseline directory is never overwritten.

In [ ]:
# Search grid: keep all non-target settings fixed for each comparison.
SEARCH_LR_GRID = [3.0e-5, 1.0e-4, 3.0e-4]
SEARCH_EFFECTIVE_BATCH_GRID = [8, 32]
SEARCH_MICRO_BATCH = 8
SEARCH_EPOCHS = 20
SEARCH_WARMUP_FRAC = 0.0
SEARCH_WEIGHT_DECAY = 0.01
SEARCH_NUM_WORKERS = 8

# Current decision: test learning rate first. This is the first new run to launch.
FINAL_DECISION = {
    'stage': 'B2 learning-rate-first search',
    'precision': 'bf16',
    'lr': 1.0e-4,
    'batch_size': SEARCH_MICRO_BATCH,
    'gradient_accumulation_steps': 1,
    'effective_batch_size': 8,
    'epochs': SEARCH_EPOCHS,
    'warmup_frac': SEARCH_WARMUP_FRAC,
    'weight_decay': SEARCH_WEIGHT_DECAY,
    'scheduler': 'cosine',
    'init_from': f'{OUT_B1}/checkpoint.pt',
    'validation_decode': ['nr3', 'nr3_rp12'],
}

def make_search_b2_config(key, lr, effective_batch):
    if effective_batch % SEARCH_MICRO_BATCH:
        raise ValueError('effective batch must be divisible by the micro batch')
    run_name = f'{BASE}__b2_search__{key}'
    path, cfg = make_config('arm_b_stage2.yaml', run_name)
    cfg['num_workers'] = SEARCH_NUM_WORKERS
    cfg['optim']['lr'] = float(lr)
    cfg['optim']['batch_size'] = SEARCH_MICRO_BATCH
    cfg['optim']['gradient_accumulation_steps'] = effective_batch // SEARCH_MICRO_BATCH
    cfg['optim']['epochs'] = SEARCH_EPOCHS
    cfg['optim']['warmup_frac'] = SEARCH_WARMUP_FRAC
    cfg['optim']['weight_decay'] = SEARCH_WEIGHT_DECAY
    with open(path, 'w') as fh:
        yaml.safe_dump(cfg, fh, sort_keys=False)
    return {
        'key': key, 'cfg_path': path, 'out_dir': cfg['output_dir'],
        'lr': float(lr), 'effective_batch_size': effective_batch,
        'precision': cfg.get('precision', 'fp32'),
        'batch_size': SEARCH_MICRO_BATCH,
        'gradient_accumulation_steps': effective_batch // SEARCH_MICRO_BATCH,
    }

# The old B2 used 3e-5 + 5% warmup and is historical reference only.
SEARCH_CONFIGS = {
    'lr3e-5_ebs8_nowarmup': make_search_b2_config('lr3e-5_ebs8_nowarmup', 3.0e-5, 8),
    'lr1e-4_ebs8': make_search_b2_config('lr1e-4_ebs8', 1.0e-4, 8),
    'lr3e-4_ebs8': make_search_b2_config('lr3e-4_ebs8', 3.0e-4, 8),
    'lr1e-4_ebs32': make_search_b2_config('lr1e-4_ebs32', 1.0e-4, 32),
}

print('=== Grid A: learning rate, fixed effective batch size 8 ===')
print('historical reference  lr=3e-5 + warmup=5%  (old B2; not part of the new grid)')
print('baseline candidate  lr=3e-5 + warmup=0  (clean no-warmup baseline)')
print('candidate  lr=1e-4  (first recommended new run)')
print('candidate  lr=3e-4  (only if the 1e-4 run is stable)')
print('=== Grid B: effective batch size, after selecting lr ===')
print('effective batch sizes:', SEARCH_EFFECTIVE_BATCH_GRID, '(micro batch=8, accumulation=1 or 4)')
print('batch candidate currently prepared: lr=1e-4, effective batch size=32')
print('\n=== FINAL_DECISION: first new run ===')
print(yaml.safe_dump(FINAL_DECISION, sort_keys=False))
print('=== Search configs created; no training was started ===')
for key, spec in SEARCH_CONFIGS.items():
    print(key, '| lr=', spec['lr'], '| effective_batch=', spec['effective_batch_size'],
          '| accumulation=', spec['gradient_accumulation_steps'], '| out=', spec['out_dir'])

## 8.2 Run one B2 search candidate

Change only `SEARCH_RUN_TO_RUN` when choosing another candidate. The first recommended value is `lr1e-4_ebs8`. This cell trains one new run from the finished B1 checkpoint, does not touch the old B2 result, and can be rerun after an interruption to resume.

In [ ]:
SEARCH_RUN_TO_RUN = 'lr1e-4_ebs8'
if SEARCH_RUN_TO_RUN not in SEARCH_CONFIGS:
    raise KeyError(f'Unknown search run: {SEARCH_RUN_TO_RUN}')
B1_CHECKPOINT_SEARCH = f'{OUT_B1}/checkpoint.pt'
if not os.path.exists(B1_CHECKPOINT_SEARCH):
    raise SystemExit(f'{B1_CHECKPOINT_SEARCH} is missing; finish B1 first.')
spec = SEARCH_CONFIGS[SEARCH_RUN_TO_RUN]
print('Selected search run:')
print(yaml.safe_dump({k: v for k, v in spec.items() if k != 'cfg_path'}, sort_keys=False))
train_one(f'Arm B2 search {SEARCH_RUN_TO_RUN}', spec['cfg_path'],
          spec['out_dir'], init_from=B1_CHECKPOINT_SEARCH, needs_metrics=True)

## 9. Run Arm B stage 1

This is the MM-WLAuslan isolated-sign adaptation stage. It has no continuous validation score. The required artifact is the finished B1 checkpoint.pt.

In [13]:
train_one('Arm B stage 1', CFG_B1, OUT_B1, needs_metrics=False)
if os.path.exists(f'{OUT_B1}/checkpoint.pt'):
    print('B1 checkpoint ready:', f'{OUT_B1}/checkpoint.pt')

arm=arm_b_stage1 device=cuda out=/content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__b1_pose
train=38580
left out 0 clips listed in /content/drive/MyDrive/auslan_work/excluded.txt

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 23042.51it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[load] missing=0 unexpected=0
trainable groups=['decoder', 'pose_encoder', 'temporal'] 587.75M / 587.75M (100.0%)
    decoder         582.40M  lr=1.00e-05
    pose_encoder      0.41M  lr=1.00e-04
    temporal          4.94M  lr=1.00e-05
batch=16 gradient_accumulation=1 effective_batch=16 updates/epoch=2411
--resume: no checkpoint in /content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__b1_pose/checkpoints; starting fresh
  {"s

## 10. Run Arm B stage 2

Run this only after B1 has produced checkpoint.pt. This stage must be Auslan-Daily after MM-WLAuslan, with all groups unfrozen.

In [14]:
B1_CHECKPOINT = f'{OUT_B1}/checkpoint.pt'
if not os.path.exists(B1_CHECKPOINT):
    raise SystemExit(f'{B1_CHECKPOINT} is missing. Finish Arm B stage 1 first.')
train_one('Arm B stage 2', CFG_B2, OUT_B2, init_from=B1_CHECKPOINT, needs_metrics=True)

arm=arm_b_stage2 device=cuda out=/content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__b2_daily
train=21998 val=1492
left out 119 clips listed in /content/drive/MyDrive/auslan_work/excluded.txt

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 18197.10it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[load] missing=0 unexpected=0
initialised from /content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__b1_pose/checkpoint.pt (arm=arm_b_stage1 step=7233)
trainable groups=['decoder', 'pose_encoder', 'temporal'] 587.75M / 587.75M (100.0%)
    decoder         582.40M  lr=3.00e-05
    pose_encoder      0.41M  lr=3.00e-05
    temporal          4.94M  lr=3.00e-05
batch=8 gradient_accumulation=1 effective_batch=8 updates/e

True

## 11. Run Arm C auxiliary-weight sweep

Each C run starts from the CSL pose-only checkpoint. If GPU time is limited, change C_WEIGHTS_TO_RUN in cell 7 to a one-element list and return later for the remaining weights. The output directories remain fixed, so rerunning skips finished weights and resumes an interrupted one.

In [ ]:
for weight in C_WEIGHTS_TO_RUN:
    if not train_one(f'Arm C aux_weight={weight}', CFG_C[weight], OUT_C[weight], needs_metrics=True):
        break

arm=arm_c device=cuda out=/content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__c__aux0.1
train=21998 aux=38580 val=1492
left out 119 clips listed in /content/drive/MyDrive/auslan_work/excluded.txt

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 22161.12it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[load] missing=0 unexpected=0
trainable groups=['decoder', 'pose_encoder', 'temporal'] 587.75M / 587.75M (100.0%)
    decoder         582.40M  lr=3.00e-05
    pose_encoder      0.41M  lr=3.00e-05
    temporal          4.94M  lr=3.00e-05
batch=8 gradient_accumulation=1 effective_batch=8 updates/epoch=2749
--resume: no checkpoint in /content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__c__aux0.1/checkpoints; star

KeyboardInterrupt: 

## 12. Re-evaluate finished B2/C runs with fixed validation decoders

Plain scores are written by training. The following two settings are additional validation-set diagnostics. Do not choose a decoder on validation and then claim that same choice is an untouched test result.

In [12]:
DECODES = {
    'nr3': {'decode.no_repeat_ngram_size': 3},
    'nr3_rp12': {
        'decode.no_repeat_ngram_size': 3,
        'decode.repetition_penalty': 1.2,
    },
}
EVAL_RUNS = [('Arm B stage 2', CFG_B2, OUT_B2)]
if 'SEARCH_CONFIGS' in globals():
    EVAL_RUNS += [
        (f'B2 search {key}', spec['cfg_path'], spec['out_dir'])
        for key, spec in SEARCH_CONFIGS.items()
    ]
EVAL_RUNS += [
    (f'Arm C aux_weight={weight}', CFG_C[weight], OUT_C[weight])
    for weight in C_WEIGHTS
]
for label, cfg_path, out_dir in EVAL_RUNS:
    if not os.path.exists(f'{out_dir}/checkpoint.pt'):
        print(f'{label}: not finished; skipping')
        continue
    for tag, settings in DECODES.items():
        eval_one(label, cfg_path, out_dir, tag, settings)

arm=arm_b_stage2 device=cuda out=/content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__b2_daily
train=21998 val=1492
left out 119 clips listed in /content/drive/MyDrive/auslan_work/excluded.txt

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 23750.02it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[load] missing=0 unexpected=0
--eval-only: weights /content/drive/MyDrive/auslan_work/runs/arm_bc__csl_stage1_weight__b2_daily/checkpoint.pt (arm=arm_b_stage2 step=54980), decode {'no_repeat_ngram_size': 3}
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message wit

## 13. Compare completed validation results

Communication and News are printed separately. Do not average them.

In [13]:
REPORT_RUNS = [('B2', OUT_B2)]
if 'SEARCH_CONFIGS' in globals():
    REPORT_RUNS += [
        (f'B2-{key}', spec['out_dir'])
        for key, spec in SEARCH_CONFIGS.items()
    ]
REPORT_RUNS += [(f'C-{weight:.1f}', OUT_C[weight]) for weight in C_WEIGHTS]
TAGS = ['plain', 'nr3', 'nr3_rp12']
FIELDS = ['BLEU-1', 'BLEU-4', 'ROUGE-L', 'unique_hyps', 'looping', 'hyp_len', 'ref_len']
found = False
for name, out_dir in REPORT_RUNS:
    for tag in TAGS:
        path = f'{out_dir}/metrics.json' if tag == 'plain' else f'{out_dir}/eval_{tag}/metrics.json'
        if not os.path.exists(path):
            continue
        found = True
        metrics = json.load(open(path))
        print(f'\n{name} | {tag}')
        for group in ('auslandaily/communication', 'auslandaily/news'):
            if group not in metrics:
                continue
            m = metrics[group]
            values = ' '.join(f'{key}={m[key]}' for key in FIELDS if key in m)
            print(f'  {group}: {values}')
if not found:
    raise SystemExit('No completed B2/C validation result found yet.')
print('\nNo subset average is reported.')


B2 | plain
  auslandaily/communication: BLEU-1=26.33 BLEU-4=3.4 ROUGE-L=12.49 unique_hyps=9.1 looping=2.1 hyp_len=4.9 ref_len=5.2
  auslandaily/news: BLEU-1=6.86 BLEU-4=0.54 ROUGE-L=6.23 unique_hyps=38.3 looping=66.6 hyp_len=29.0 ref_len=15.9

B2 | nr3
  auslandaily/communication: BLEU-1=26.28 BLEU-4=3.43 ROUGE-L=12.53 unique_hyps=8.7 looping=0.0 hyp_len=4.7 ref_len=5.2
  auslandaily/news: BLEU-1=18.25 BLEU-4=1.21 ROUGE-L=9.62 unique_hyps=34.3 looping=0.0 hyp_len=16.2 ref_len=15.9

B2 | nr3_rp12
  auslandaily/communication: BLEU-1=26.68 BLEU-4=3.53 ROUGE-L=12.86 unique_hyps=10.1 looping=0.0 hyp_len=4.6 ref_len=5.2
  auslandaily/news: BLEU-1=18.25 BLEU-4=1.18 ROUGE-L=9.34 unique_hyps=35.7 looping=0.0 hyp_len=16.8 ref_len=15.9

No subset average is reported.


## After this notebook

Use B2 and the C sweep as the next comparison against the Arm A references. If one setting is selected for final reporting, fix it before evaluating the complete test split and do not apply excluded.txt to that final test score. Test_ITW, Test_TED, Test_SYN, and Test_MTV remain out of training.